# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

### Question 1 — Target Distribution

What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [3]:
# Check the distribution of the target variable
print(fraud["fraud"].value_counts())

# Calculate the percentage of each class
print("\nPercentages:")
print(fraud["fraud"].value_counts(normalize=True) * 100)

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

Percentages:
fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64


### Interpretation — Target Distribution

The dataset contains 1,000,000 transactions.

- **91.26%** are non-fraudulent transactions (class 0).
- **8.74%** are fraudulent transactions (class 1).

The dataset is imbalanced because fraudulent transactions represent a small minority.

This imbalance can affect the model's performance. A model may achieve high accuracy by predicting mostly non-fraudulent transactions, while failing to detect actual fraud.

Therefore, accuracy alone is not enough. We should also consider precision, recall, and F1-score.

### Question 2 — Logistic Regression

Train a LogisticRegression model using the original imbalanced dataset.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Separate features and target
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=17,
    stratify=y)

# Train the model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


### Question 3 — Model Evaluation

Evaluate the Logistic Regression model using accuracy, precision, recall, F1-score, and the confusion matrix.

In [5]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix)

# Make predictions
y_pred = log_model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.959065

Confusion Matrix:
[[181276   1243]
 [  6944  10537]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.89      0.60      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



### Interpretation — Logistic Regression Evaluation

The Logistic Regression model achieves an accuracy of 95.91%.

However, accuracy does not tell the whole story because the dataset is imbalanced.

For fraudulent transactions (class 1):

- **Precision: 89%** — Most transactions predicted as fraud are actually fraudulent.
- **Recall: 60%** — The model detects only 60% of actual fraud cases.
- **F1-score: 72%** — The model has moderate performance when balancing precision and recall.

The confusion matrix shows that 6,944 fraudulent transactions were incorrectly classified as non-fraudulent.

**Conclusion:** Despite its high accuracy, the model misses too many fraud cases. We will explore resampling techniques to improve fraud detection.

### Question 4 — Random Oversampling

Apply random oversampling to balance the training dataset.

Train a new Logistic Regression model using the balanced data and compare its performance with the original model.

In [6]:
from imblearn.over_sampling import RandomOverSampler

# Apply oversampling to the training data only
oversampler = RandomOverSampler(random_state=17)

X_train_over, y_train_over = oversampler.fit_resample(
    X_train, y_train)

# Check the new class distribution
print("Original distribution:")
print(y_train.value_counts())

print("\nAfter oversampling:")
print(y_train_over.value_counts())

Original distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After oversampling:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


### Interpretation — Random Oversampling

Before oversampling, the training dataset contained:

- 730,078 non-fraudulent transactions.
- 69,922 fraudulent transactions.

After random oversampling, both classes contain 730,078 observations.

Random oversampling balances the dataset by randomly duplicating observations from the minority class.

The training dataset is now balanced (50% / 50%).

However, duplicating observations may increase the risk of overfitting.

### Random Oversampling — Model Evaluation

Train a Logistic Regression model on the oversampled training data.

Evaluate its performance on the original test set and compare the results with the baseline model.

In [7]:
# Train Logistic Regression on oversampled data
log_model_over = LogisticRegression(max_iter=1000)

log_model_over.fit(X_train_over, y_train_over)

# Predict on the original test set
y_pred_over = log_model_over.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_over))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_over))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_over))

Accuracy: 0.93419

Confusion Matrix:
[[170236  12283]
 [   879  16602]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.57      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



### Interpretation — Oversampling Model

Random oversampling significantly improves fraud detection.

- **Recall increases from 60% to 95%.**
- Missed fraud cases decrease from 6,944 to 879.
- Accuracy decreases from 95.91% to 93.42%.
- Precision decreases from 89% to 57%.
- False positives increase from 1,243 to 12,283.

The model detects more fraudulent transactions but also generates more false alarms.

**Conclusion:** Oversampling is useful when detecting fraud is more important than avoiding false positives. However, the best model depends on the cost of missed fraud and false alarms.

### Random Undersampling

Apply random undersampling to balance the training dataset by reducing the majority class.

In [8]:
from imblearn.under_sampling import RandomUnderSampler

# Apply undersampling to training data only
undersampler = RandomUnderSampler(random_state=17)

X_train_under, y_train_under = undersampler.fit_resample(
    X_train, y_train
)

# Check class distribution
print("Original distribution:")
print(y_train.value_counts())

print("\nAfter undersampling:")
print(y_train_under.value_counts())

Original distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After undersampling:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


### Interpretation — Random Undersampling

Before undersampling, the training dataset contained:

- 730,078 non-fraudulent transactions.
- 69,922 fraudulent transactions.

After undersampling, both classes contain 69,922 observations.

Random undersampling balances the dataset by randomly removing observations from the majority class.

The main advantage is a smaller training dataset, which makes training faster.

However, removing observations may lead to a loss of useful information.

### Random Undersampling — Model Evaluation

Train a Logistic Regression model using the undersampled training data.

Evaluate its performance on the original test set and compare the results with the baseline and oversampling models.

In [9]:
# Train Logistic Regression on undersampled data
log_model_under = LogisticRegression(max_iter=1000)

log_model_under.fit(X_train_under, y_train_under)

# Predict on the original test set
y_pred_under = log_model_under.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_under))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_under))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_under))

Accuracy: 0.934365

Confusion Matrix:
[[170254  12265]
 [   862  16619]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



### Interpretation — Undersampling Model

Random undersampling achieves an accuracy of 93.44%.

For fraudulent transactions (class 1):

- Precision: 58%.
- Recall: 95%.
- F1-score: 72%.
- Missed fraud cases: 862.
- False positives: 12,265.

The results are very similar to random oversampling.

Undersampling detects slightly more fraud cases and generates slightly fewer false positives.

However, it removes many non-fraudulent observations, which may result in a loss of useful information.

**Conclusion:** Undersampling is a useful alternative to oversampling, especially when reducing training time is important.

### SMOTE — Synthetic Minority Oversampling

Apply SMOTE to balance the training dataset by generating synthetic observations for the minority class.

In [10]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to training data only
smote = SMOTE(random_state=17)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Check the new class distribution
print("Original distribution:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Original distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After SMOTE:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


### Interpretation — SMOTE

Before SMOTE, the training dataset contained:

- 730,078 non-fraudulent transactions.
- 69,922 fraudulent transactions.

After SMOTE, both classes contain 730,078 observations.

Unlike random oversampling, SMOTE creates synthetic minority class observations instead of simply duplicating existing ones.

This technique helps balance the dataset while introducing new synthetic examples.

However, synthetic observations may not always represent realistic transactions.

### SMOTE — Model Evaluation

Train a Logistic Regression model using the SMOTE-balanced training data.

Evaluate its performance on the original test set and compare the results with the previous models.

In [11]:
# Train Logistic Regression on SMOTE data
log_model_smote = LogisticRegression(max_iter=1000)

log_model_smote.fit(X_train_smote, y_train_smote)

# Predict on the original test set
y_pred_smote = log_model_smote.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_smote))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

Accuracy: 0.93472

Confusion Matrix:
[[170361  12158]
 [   898  16583]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



### Interpretation — SMOTE Model

The SMOTE model achieves an accuracy of 93.47%.

For fraudulent transactions (class 1):

- Precision: 58%.
- Recall: 95%.
- F1-score: 72%.
- Missed fraud cases: 898.
- False positives: 12,158.

SMOTE significantly improves fraud detection compared to the baseline model.

It achieves slightly higher accuracy than random oversampling and undersampling, while generating fewer false positives.

However, it misses slightly more fraudulent transactions than the other resampling methods.

### Final Comparison — Imbalanced Classification

We trained four Logistic Regression models:

1. Baseline model without resampling
2. Random Oversampling
3. Random Undersampling
4. SMOTE

The baseline model achieved the highest accuracy (95.91%) and precision (89%), but detected only 60% of fraudulent transactions.

All three resampling techniques increased fraud recall to approximately 95%, significantly reducing the number of missed fraud cases.

Random undersampling missed the fewest fraud cases (862), while SMOTE generated the fewest false positives among the resampled models (12,158).

The F1-score remained approximately 72% for all four models.

**Conclusion:** No model is best for every situation. If detecting as many fraud cases as possible is the priority, random undersampling is slightly better in this experiment. If reducing false alarms is more important, SMOTE offers a small advantage among the resampling methods.

The final choice should depend on the business cost of missed fraud and false positives.